# Iris Species Classification
**Student ID:** st126112  
**Algorithm:** Random Forest Classifier  
**Dataset:** Iris (sklearn built-in)

This notebook covers:
1. Data loading & exploration
2. Exploratory Data Analysis (EDA)
3. Model training (Random Forest)
4. Evaluation (accuracy, classification report, confusion matrix)
5. Model persistence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

sns.set_theme(style='whitegrid')
print("Libraries loaded successfully.")

## 1. Data Loading & Exploration

In [ ]:
# Load Iris dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:\n{df['species'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Feature distributions by species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = iris.feature_names

for ax, feature in zip(axes.flatten(), features):
    for species in iris.target_names:
        subset = df[df['species'] == species][feature]
        ax.hist(subset, alpha=0.6, label=species, bins=15)
    ax.set_title(feature)
    ax.set_xlabel('cm')
    ax.legend()

plt.suptitle('Feature Distributions by Species', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot to visualise inter-feature relationships
sns.pairplot(df, hue='species', diag_kind='kde')
plt.suptitle('Pairplot of Iris Features', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.drop(columns='species').corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

## 3. Model Training

In [ ]:
X = iris.data
y = iris.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)
model.fit(X_train, y_train)
print(f"Training complete. Train samples: {len(X_train)}, Val samples: {len(X_val)}")

## 4. Evaluation

In [ ]:
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
importances = pd.Series(model.feature_importances_, index=iris.feature_names).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(7, 4), color='steelblue')
plt.title('Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Save Model

In [ ]:
joblib.dump(model, 'model.pkl')
print("Model saved to model.pkl")
print(f"Model type: {type(model).__name__}")
print(f"Number of estimators: {model.n_estimators}")
print(f"Max depth: {model.max_depth}")
print(f"Validation accuracy: {accuracy:.4f}")